# C: Logistic regression: infrastructure proximity and conflict risk

We estimate a logistic regression on all 383,693 hexes in the corridor. The outcome is binary: whether a hex recorded at least one mining-related ACLED event between 2015 and 2025. Predictors are log-transformed distances to the nearest railway, major road, and town, along with mining intensity (log1p of Liu Ha_erased), ASM share, and LSM share. Distance variables are log10-transformed because their raw distributions are strongly right-skewed (see Figure 3). Country fixed effects are included via dummies.

In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "submission" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

hex_features = gpd.read_file(PROCESSED_DIR / "hex_features.gpkg")
print(f"{len(hex_features):,} hexes loaded")

hex_features["log_liu"]       = np.log1p(hex_features["liu_ha_erased"])
hex_features["log_dist_rail"] = np.log10(1 + hex_features["dist_rail_km"])
hex_features["log_dist_road"] = np.log10(1 + hex_features["dist_road_km"])
hex_features["log_dist_town"] = np.log10(1 + hex_features["dist_town_km"])
hex_features["any_incident"]  = (hex_features["n_inc_mining_total"] > 0).astype(int)

base_rate = hex_features["any_incident"].mean()
print(f"Outcome base rate: {base_rate:.4%}  ({int(hex_features['any_incident'].sum()):,} of {len(hex_features):,})")

383,693 hexes loaded
Outcome base rate: 0.0365%  (140 of 383,693)


## Model

A negative coefficient on a distance term means hexes closer to that feature are more likely to have recorded an incident.

In [3]:
model = smf.glm(
    "any_incident ~ log_liu + log_dist_rail + log_dist_road + log_dist_town "
    "+ asm_share + lsm_share + C(country)",
    data=hex_features,
    family=sm.families.Binomial(),
).fit()

print(f"N = {int(model.nobs):,}")
print(f"Pseudo-R² (McFadden) = {1 - model.llf / model.llnull:.3f}")

N = 383,693
Pseudo-R² (McFadden) = 0.270


## Table 1: Regression coefficients

In [4]:
terms = [
    ("log_liu",       r"log1p(mine area)"),
    ("log_dist_rail", r"log(1 + d_rail)"),
    ("log_dist_road", r"log(1 + d_road)"),
    ("log_dist_town", r"log(1 + d_town)"),
    ("asm_share",     "ASM share"),
    ("lsm_share",     "LSM share"),
]

rows = []
for key, label in terms:
    b = model.params.get(key, np.nan)
    p = model.pvalues.get(key, np.nan)
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    rows.append({"Term": label, "β̂": round(b, 3), "p": round(p, 4), "": stars})

table = pd.DataFrame(rows).set_index("Term")
print(table.to_string())
print(f"\nN = {int(model.nobs):,}   outcome base rate: {base_rate:.2%}   country dummies included")
print("** p < 0.01, *** p < 0.001")

                     β̂       p     
Term                                
log1p(mine area)  0.425  0.0000  ***
log(1 + d_rail)  -0.397  0.0036   **
log(1 + d_road)  -0.687  0.0003  ***
log(1 + d_town)  -3.808  0.0000  ***
ASM share         0.329  0.7610     
LSM share         0.124  0.8898     

N = 383,693   outcome base rate: 0.04%   country dummies included
** p < 0.01, *** p < 0.001


In [5]:
# Save full results
rows_full = []
for term in model.params.index:
    rows_full.append({
        "term":    term,
        "coef":    float(model.params[term]),
        "std_err": float(model.bse[term]),
        "p":       float(model.pvalues[term]),
    })
pd.DataFrame(rows_full).to_csv(PROCESSED_DIR / "logistic_supplement.csv", index=False)
print(f"Saved {PROCESSED_DIR / 'logistic_supplement.csv'}")

Saved /Users/inge/Desktop/ITU/GeoDS/gds/data/processed/logistic_supplement.csv
